# Deep Learning sur TF-IDF — MLP (Multi-Layer Perceptron)

MLP PyTorch sur features TF-IDF pour la classification de sentiment Yelp.

**Architecture** : Input(10000) → Dense(256) → ReLU → Dropout → Dense(128) → ReLU → Dropout → Output

**Deux tâches** : Polarité (3 classes) et Score (1-5 étoiles)

In [ ]:
import sys
sys.path.insert(0, '../..')

import os
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

from src.constants import POLARITY_NAMES, SCORE_NAMES
from src.ml_utils import load_and_prepare, split_data
from src.dl_utils import get_device, set_seed, make_loaders, train_model, evaluate_model
from src.evaluation import plot_confusion, plot_training_curves, print_report
from src import setup_plot_style

setup_plot_style()
set_seed()
device = get_device()
MODELS_DIR = '../../models'
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Device : {device}")

## 1. Chargement et Préparation

In [ ]:
df = load_and_prepare()
data = split_data(df['text'], df['polarity'], df['stars'])
print(f"Train: {len(data['X_train'])} | Val: {len(data['X_val'])} | Test: {len(data['X_test'])}")

## 2. Vectorisation TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(max_features=10_000, min_df=5, max_df=0.7, ngram_range=(1, 2))

X_train = vectorizer.fit_transform(data['X_train']).toarray()
X_val = vectorizer.transform(data['X_val']).toarray()
X_test = vectorizer.transform(data['X_test']).toarray()

# Score 0-indexed pour PyTorch (1-5 → 0-4)
y_sc_train = (data['y_sc_train'] - 1).values.astype(int)
y_sc_val = (data['y_sc_val'] - 1).values.astype(int)
y_sc_test = (data['y_sc_test'] - 1).values.astype(int)

print(f"Dimension TF-IDF : {X_train.shape[1]} features")

## 3. Architecture MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model_test = MLP(X_train.shape[1], 3)
print(model_test)
print(f"\nParamètres : {sum(p.numel() for p in model_test.parameters()):,}")
del model_test

## 4. Polarité (3 classes)

In [ ]:
train_pol, val_pol, test_pol = make_loaders(
    X_train, data['y_pol_train'].values,
    X_val, data['y_pol_val'].values,
    X_test, data['y_pol_test'].values
)

model_pol = MLP(X_train.shape[1], num_classes=3)
model_pol, history_pol = train_model(model_pol, train_pol, val_pol, device=device)

In [ ]:
plot_training_curves(history_pol, 'MLP TF-IDF — Polarité')

preds_pol, labels_pol = evaluate_model(model_pol, test_pol, device=device)
print_report(labels_pol, preds_pol, POLARITY_NAMES, 'MLP TF-IDF — Polarité (Test)')
plot_confusion(labels_pol, preds_pol, POLARITY_NAMES, 'MLP TF-IDF — Polarité')

## 5. Score (1-5 étoiles)

In [ ]:
train_sc, val_sc, test_sc = make_loaders(
    X_train, y_sc_train, X_val, y_sc_val, X_test, y_sc_test
)

model_score = MLP(X_train.shape[1], num_classes=5)
model_score, history_score = train_model(model_score, train_sc, val_sc, device=device)

In [ ]:
plot_training_curves(history_score, 'MLP TF-IDF — Score')

preds_sc, labels_sc = evaluate_model(model_score, test_sc, device=device)
# Reconvertir en 1-5 pour l'affichage
print_report(labels_sc, preds_sc, SCORE_NAMES, 'MLP TF-IDF — Score (Test)')
plot_confusion(labels_sc, preds_sc, SCORE_NAMES, 'MLP TF-IDF — Score')

## 6. Sauvegarde

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

comparison = pd.DataFrame({
    'Tâche': ['Polarité', 'Score'],
    'Accuracy': [accuracy_score(labels_pol, preds_pol), accuracy_score(labels_sc, preds_sc)],
    'F1 Macro': [f1_score(labels_pol, preds_pol, average='macro'),
                 f1_score(labels_sc, preds_sc, average='macro')],
})
print("=== RÉSUMÉ MLP TF-IDF ===")
display(comparison)

torch.save(model_pol.state_dict(), os.path.join(MODELS_DIR, 'mlp_tfidf_polarity.pt'))
torch.save(model_score.state_dict(), os.path.join(MODELS_DIR, 'mlp_tfidf_score.pt'))
print(f"\nModèles sauvegardés dans {MODELS_DIR}")